In [1]:
import requests
import getpass
import numpy as np
import pandas as pd
import geopandas as gpd
from pathlib import Path

_user = getpass.getuser()

In [2]:
data_dir = Path(
    f"/Users/{_user}/Library/CloudStorage/Box-Box/DSA Projects/Spatial Analysis and Mapping/CalEnviroScreen/Data"
)
nhgis_crosswalk_path = data_dir / "nhgis_tr2010_tr2020_06" / "nhgis_tr2010_tr2020_06.csv"
ces_4_path = data_dir / "CalEnviroScreen_4_0_Results__5547037270108105365.xlsx"
ces_5_path = data_dir / "calenviroscreen50results_d_12226.xlsx"
# ces_5_geo_path = data_dir / "calenviroscreen50_D_12226.gdb"
epc_2018 = "https://services3.arcgis.com/i2dkYWmb4wHvYPda/arcgis/rest/services/equity_priority_communities_2025_acs2018/FeatureServer/0/query?outFields=*&where=epc_2050=1&f=geojson"
epc_2022 = "https://services3.arcgis.com/i2dkYWmb4wHvYPda/arcgis/rest/services/draft_equity_priority_communities_pba2050plus_acs2022a/FeatureServer/0/query?outFields=*&where=epc_2050p=1&f=geojson"

In [3]:
# read in data
crosswalk_df = pd.read_csv(nhgis_crosswalk_path, dtype={"tr2010ge": str, "tr2020ge": str})
ces_4_df = pd.read_excel(ces_4_path, sheet_name=0)
ces_5_df = pd.read_excel(ces_5_path, sheet_name=0)
# ces_5_gdf = gpd.read_file(ces_5_path, layer="CES5_Draft")
epc_2018_gdf = gpd.read_file(epc_2018)
epc_2022_gdf = gpd.read_file(epc_2022)

/Users/jcroff/anaconda3/envs/basis-env/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [4]:
# fix ces4 and ces5 tract id columns to match crosswalk
# tract id leading zeros were dropped in the excel files, so we need to add them back
ces_4_df["tract"] = ces_4_df["tract"].astype(str).str.zfill(11)
ces_5_df["Census Tract"] = ces_5_df["Census Tract"].astype(str).str.zfill(11)
# ces_5_gdf["tract"] = ces_5_gdf["tract"].astype(int).astype(str).str.zfill(11)

In [5]:
# rename all version of geoid in all dataframes to tract_geoid
ces_4_df = ces_4_df.rename(columns={"tract": "tract_geoid"})
ces_5_df = ces_5_df.rename(columns={"Census Tract": "tract_geoid"})
# ces_5_gdf = ces_5_gdf.rename(columns={"tract": "tract_geoid"})
epc_2018_gdf = epc_2018_gdf.rename(columns={"geoid": "tract_geoid"})

In [6]:
# Pull all California county names and FIPS codes from Census API
url = "https://api.census.gov/data/2020/dec/pl?get=NAME&for=county:*&in=state:06"
response = requests.get(url)
counties = pd.DataFrame(response.json()[1:], columns=['name', 'state', 'county'])

# Create the 5-digit county GEOID to match what you'll derive from the crosswalk
counties['county_geoid'] = counties['state'] + counties['county']

# Clean up the name — Census returns "Alameda County, California"
counties['county_name'] = counties['name'].str.replace(', California', '')

# Remove "County" from the name to match what CES uses
counties['county_name'] = counties['county_name'].str.replace(' County', '')

In [7]:
# Pull 2020 and 2010 CA tracts as geospatial data (GeoDataFrames) from Census TIGER
tracts_2020_url = "https://www2.census.gov/geo/tiger/TIGER2020/TRACT/tl_2020_06_tract.zip"
tracts_2010_url = "https://www2.census.gov/geo/tiger/TIGER2010/TRACT/2010/tl_2010_06_tract10.zip"

ca_tracts_2020 = gpd.read_file(tracts_2020_url)
ca_tracts_2010 = gpd.read_file(tracts_2010_url)

def _pick_col(df: pd.DataFrame, candidates: list[str]) -> str:
    """Return the first matching column (case-insensitive) from candidates."""
    lookup = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lookup:
            return lookup[cand.lower()]
    raise KeyError(f"None of {candidates} found in columns: {list(df.columns)}")

# Map source columns to consistent names used downstream
col_map_2020 = {
    _pick_col(ca_tracts_2020, ["NAMELSAD20", "NAME20", "NAMELSAD", "NAME"]): "name",
    _pick_col(ca_tracts_2020, ["STATEFP20", "STATEFP"]): "state",
    _pick_col(ca_tracts_2020, ["COUNTYFP20", "COUNTYFP"]): "county",
    _pick_col(ca_tracts_2020, ["TRACTCE20", "TRACTCE"]): "tract",
    _pick_col(ca_tracts_2020, ["GEOID20", "GEOID"]): "tract_geoid",
}
col_map_2010 = {
    _pick_col(ca_tracts_2010, ["NAMELSAD10", "NAME10", "NAMELSAD", "NAME"]): "name",
    _pick_col(ca_tracts_2010, ["STATEFP10", "STATEFP"]): "state",
    _pick_col(ca_tracts_2010, ["COUNTYFP10", "COUNTYFP"]): "county",
    _pick_col(ca_tracts_2010, ["TRACTCE10", "TRACTCE"]): "tract",
    _pick_col(ca_tracts_2010, ["GEOID10", "GEOID"]): "tract_geoid",
}

ca_tracts_2020 = ca_tracts_2020.rename(columns=col_map_2020)
ca_tracts_2010 = ca_tracts_2010.rename(columns=col_map_2010)

# Keep only fields needed downstream plus geometry
ca_tracts_2020 = ca_tracts_2020[["name", "state", "county", "tract", "tract_geoid", "geometry"]].copy()
ca_tracts_2010 = ca_tracts_2010[["name", "state", "county", "tract", "tract_geoid", "geometry"]].copy()

# Normalize CRS for easier downstream mapping/export
ca_tracts_2020 = ca_tracts_2020.to_crs(epsg=4326)
ca_tracts_2010 = ca_tracts_2010.to_crs(epsg=4326)

print("2020 CA tracts (geospatial):", len(ca_tracts_2020))
print("2010 CA tracts (geospatial):", len(ca_tracts_2010))

2020 CA tracts (geospatial): 9129
2010 CA tracts (geospatial): 8057


In [8]:
# Create a county_geoid column in the crosswalk to match with the counties df
crosswalk_df["county_geoid"] = crosswalk_df["tr2020ge"].str[:5]

# map county names from counties df to the crosswalk df
crosswalk_df["county_name"] = crosswalk_df["county_geoid"].map(
    counties.set_index("county_geoid")["county_name"]
)

# map county names from counties df to the ca_tracts_2020 df
ca_tracts_2020["county_geoid"] = ca_tracts_2020["tract_geoid"].str[:5]
ca_tracts_2020["county_name"] = ca_tracts_2020["county_geoid"].map(
    counties.set_index("county_geoid")["county_name"]
)

# map county names from counties df to the ca_tracts_2010 df
ca_tracts_2010["county_geoid"] = ca_tracts_2010["tract_geoid"].str[:5]
ca_tracts_2010["county_name"] = ca_tracts_2010["county_geoid"].map(
    counties.set_index("county_geoid")["county_name"]       
)

In [9]:
# create a disadvantaged communities flag for ces 4 and 5 based on the top 25% cutoff in each version
ces_4_df["ces4_dac"] = np.where(ces_4_df["CIscoreP"] >= 75, 1, 0)
ces_5_df["ces5_dac"] = np.where(ces_5_df["Draft CES 5.0 Percentile"] >= 75, 1, 0)
# ces_5_gdf["ces5_dac"] = np.where(ces_5_gdf["CIscore_Pctl"] >= 75, 1, 0)

In [10]:
# Do a sanity check to ensure that when wt_pop is grouped by tr2010ge, that the sum is not greater than 1
sums = crosswalk_df.groupby("tr2010ge")["wt_pop"].sum()
print(sums.min(), sums.max())
if not sums.between(0, 1.01).all():
    raise ValueError(f"Unexpected weight sums: min={sums.min()}, max={sums.max()}")

0.0 1.0000000001


In [11]:
# Create a 2010 to 2020 tract relationship classification
# Here is the logic:
# If the count of 2020 tracts (tr2020ge) = 1 and the count of 2010 tracts (tr2010ge) = 1, then it's a "1-to-1" relationship
# If the count of 2020 tracts (tr2020ge) > 1 and the count of 2010 tracts (tr2010ge) = 1, then it's a "split" (one old tract became many new tracts)
# If the count of 2020 tracts (tr2020ge) = 1 and the count of 2010 tracts (tr2010ge) > 1, then it's a "merge" (many old tracts became one new tract)
# If the count of 2020 tracts (tr2020ge) > 1 and the count of 2010 tracts (tr2010ge) > 1, then it's a "complex" relationship (e.g., some combination of splits and merges)

# tract_ct_2020: how many 2020 tracts each 2010 tract maps to
# tract_ct_2010: how many 2010 tracts map to each 2020 tract
tract_ct_2020 = crosswalk_df.groupby("tr2010ge")["tr2020ge"].nunique()
tract_ct_2010 = crosswalk_df.groupby("tr2020ge")["tr2010ge"].nunique()

crosswalk_df["tract_ct_2020"] = crosswalk_df["tr2010ge"].map(tract_ct_2020)
crosswalk_df["tract_ct_2010"] = crosswalk_df["tr2020ge"].map(tract_ct_2010)

conditions = [
    (crosswalk_df["tract_ct_2020"] == 1) & (crosswalk_df["tract_ct_2010"] == 1),
    (crosswalk_df["tract_ct_2020"] > 1) & (crosswalk_df["tract_ct_2010"] == 1),
    (crosswalk_df["tract_ct_2020"] == 1) & (crosswalk_df["tract_ct_2010"] > 1),
    (crosswalk_df["tract_ct_2020"] > 1) & (crosswalk_df["tract_ct_2010"] > 1),
]
labels = ["1-to-1", "split", "merge", "complex"]
crosswalk_df["tract_relationship"] = np.select(condlist=conditions, choicelist=labels, default=None)

assert crosswalk_df["tract_relationship"].isna().sum() == 0, "Unclassified rows found"
crosswalk_df["tract_relationship"].value_counts()

tract_relationship
complex    5405
1-to-1     4023
split      2513
merge       954
Name: count, dtype: int64

In [12]:
def carry_forward_flag(
    crosswalk_df: pd.DataFrame,
    source_df: pd.DataFrame | gpd.GeoDataFrame,
    source_geoid_col: str,
    flag_col: str,
    threshold: float = 0.5,
) -> pd.DataFrame:
    """Carries a binary flag from 2010 census tracts forward to 2020 census tracts
    using population-weighted crosswalk weights.

    For each 2020 tract, sums the weighted contributions from all source 2010 tracts
    (flag * wt_pop) and applies a threshold to determine the forward flag.

    Args:
        crosswalk_df: Crosswalk with columns: tr2010ge, tr2020ge, wt_pop, tract_ct_2010.
        source_df: Source dataset containing the flag to carry forward.
        source_geoid_col: Name of the tract GEOID column in source_df
            (must match tr2010ge format).
        flag_col: Name of the binary (0/1) flag column to carry forward.
        threshold: Minimum weighted population fraction to assign a forward
            flag of 1. Defaults to 0.5.

    Returns:
        DataFrame with columns tr2020ge, tract_ct_2010, and {flag_col}_forward
        for all 2020 tracts in the crosswalk.
    """
    all_2020_tracts = crosswalk_df["tr2020ge"].unique()

    merged = crosswalk_df[["tr2010ge", "tr2020ge", "wt_pop"]].merge(
        source_df[[source_geoid_col, flag_col]],
        left_on="tr2010ge",
        right_on=source_geoid_col,
        how="left",
    )
    merged[flag_col] = merged[flag_col].fillna(0)
    merged["flag_weighted"] = merged[flag_col] * merged["wt_pop"]

    weighted_sum = (
        merged.groupby("tr2020ge")["flag_weighted"].sum().reindex(all_2020_tracts, fill_value=0)
    )
    forward_flag = (weighted_sum >= threshold).astype(int).rename(f"{flag_col}_forward")

    # tract_ct_2010 is constant per tr2020ge — take first value per destination tract
    tract_ct_2010 = (
        crosswalk_df.drop_duplicates("tr2020ge")
        .set_index("tr2020ge")["tract_ct_2010"]
        .reindex(all_2020_tracts)
    )

    return pd.DataFrame(
        {"tract_ct_2010": tract_ct_2010, f"{flag_col}_forward": forward_flag}
    ).reset_index(names="tr2020ge")

In [13]:
ces4_dac_forward = carry_forward_flag(
    crosswalk_df=crosswalk_df,
    source_df=ces_4_df,
    source_geoid_col="tract_geoid",
    flag_col="ces4_dac",
    threshold=0.5,
)
epc_2018_forward = carry_forward_flag(
    crosswalk_df=crosswalk_df,
    source_df=epc_2018_gdf,
    source_geoid_col="tract_geoid",
    flag_col="epc_2050",
    threshold=0.5,
)

print(ces4_dac_forward.value_counts())
print(epc_2018_forward.value_counts())

tr2020ge     tract_ct_2010  ces4_dac_forward
06001400100  8              0                   1
06001422700  2              0                   1
06001423700  2              0                   1
06001423800  6              0                   1
06001982100  5              0                   1
                                               ..
06115040901  2              0                   1
06115041001  1              0                   1
06115041002  1              0                   1
06115041101  1              0                   1
06115041102  1              0                   1
Name: count, Length: 9129, dtype: int64
tr2020ge     tract_ct_2010  epc_2050_forward
06001400100  8              0                   1
06001422700  2              0                   1
06001423700  2              0                   1
06001423800  6              0                   1
06001982100  5              0                   1
                                               ..
06115040901  2      

In [14]:
# join the forward carried flags to the CES5 dataframe, which contains all 2020 tracts for California
ces4_dac_forward = ces4_dac_forward.rename(columns={"tr2020ge": "tract_geoid"})
epc_2018_forward = epc_2018_forward.rename(columns={"tr2020ge": "tract_geoid"})

ces_5_cols = ["tract_geoid", "ces5_dac"]
ces_4_cols = ["tract_geoid", "ces4_dac_forward"]
epc_2018_cols = ["tract_geoid", "epc_2050_forward"]

# merge CES5 with the CA tracts table to bring in county_name from Census API tracts
comparison_2020_df = pd.merge(
    ca_tracts_2020[["tract_geoid", "county_name"]],
    ces_5_df[ces_5_cols],
    on="tract_geoid",
    how="left",
)

# merge CES5 with CES4 forward-carried flag
comparison_2020_df = pd.merge(
    comparison_2020_df, ces4_dac_forward[ces_4_cols], on="tract_geoid", how="left"
)

# merge 2018 EPC forward flag with CES5
comparison_2020_df = pd.merge(
    comparison_2020_df, epc_2018_forward[epc_2018_cols], on="tract_geoid", how="left"
)

# merge 2022 EPC flag with CES5
comparison_2020_df = pd.merge(
    comparison_2020_df, epc_2022_gdf[["tract_geoid", "epc_2050p"]], on="tract_geoid", how="left"
)

# fill NA values in ces4_dac_forward, epc_2050_forward, epc_2050p with 0
comparison_2020_df["ces5_dac"] = comparison_2020_df["ces5_dac"].fillna(0).astype(int)

comparison_2020_df["ces4_dac_forward"] = (
    comparison_2020_df["ces4_dac_forward"].fillna(0).astype(int)
)
comparison_2020_df["epc_2050_forward"] = (
    comparison_2020_df["epc_2050_forward"].fillna(0).astype(int)
)
comparison_2020_df["epc_2050p"] = comparison_2020_df["epc_2050p"].fillna(0).astype(int)

# rename columns for clarity; keep census-derived county_name as the canonical county field
comparison_2020_df = comparison_2020_df.rename(
    columns={
        "ces4_dac_forward": "ces4_dac_2010_to_2020",
        "epc_2050_forward": "epc2050_2010_to_2020",
        "epc_2050p": "epc2050p",
    }
)

# Classify Bay Area status in the comparison df
bay_area_counties = [
    "Alameda",
    "Contra Costa",
    "Marin",
    "Napa",
    "San Francisco",
    "San Mateo",
    "Santa Clara",
    "Solano",
    "Sonoma",
]

comparison_2020_df["bay_area_class"] = np.where(
    comparison_2020_df["county_name"].isin(bay_area_counties), "Bay Area", "Outside Bay Area"
)

comparison_2020_df.head()

,tract_geoid,county_name,ces5_dac,ces4_dac_2010_to_2020,epc2050_2010_to_2020,epc2050p,bay_area_class
0,06029004402,Kern,1,1,0,0,Outside Bay Area
1,06047000802,Merced,1,1,0,0,Outside Bay Area
2,06085501402,Santa Clara,0,0,1,0,Bay Area
3,06005000102,Amador,0,0,0,0,Outside Bay Area
4,06029004901,Kern,1,1,0,0,Outside Bay Area


In [34]:
# Comparison-based summaries using the same pivot layout
def make_bay_area_pivot(df: pd.DataFrame, flag_col: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    summary = (
        df.groupby("bay_area_class")[flag_col]
        .agg(
            total_dac="sum",
            total_not_dac=lambda s: (s == 0).sum(),
            total_tracts="count",
        )
        .reset_index()
)
    summary["dac_pct"] = summary["total_dac"] / summary["total_tracts"] * 100

    pivot = (
        summary.set_index("bay_area_class")[["total_dac", "total_not_dac"]]
        .T
        .reindex(columns=["Bay Area", "Outside Bay Area"], fill_value=0)
        .astype(int)
)
    pivot["total Tracts"] = pivot["Bay Area"] + pivot["Outside Bay Area"]
    return summary, pivot

comparison_ces4_summary, comparison_ces4_pivot = make_bay_area_pivot(
    comparison_2020_df, "ces4_dac_2010_to_2020" ,
)
comparison_ces5_summary, comparison_ces5_pivot = make_bay_area_pivot(
    comparison_2020_df, "ces5_dac" ,
)

print("Comparison 2020 CES4-forward pivot")
display(comparison_ces4_pivot)
print("Comparison 2020 CES5 pivot")
display(comparison_ces5_pivot)

Comparison 2020 CES4-forward pivot


bay_area_class,Bay Area,Outside Bay Area,total Tracts
total_dac,113,1827,1940
total_not_dac,1659,5530,7189


Comparison 2020 CES5 pivot


bay_area_class,Bay Area,Outside Bay Area,total Tracts
total_dac,122,2136,2258
total_not_dac,1650,5221,6871


## Create simple summary stats of CES4 and CES5

In [32]:
# join all 2010 CA tracts with CES4 so denominator includes every 2010 tract
ces_4_gdf = ca_tracts_2010.merge(
    ces_4_df[["tract_geoid", "ces4_dac"]],
    on="tract_geoid",
    how="left",
)

# treat tracts missing from CES4 as non-DAC for full-tract denominator summaries
ces_4_gdf["ces4_dac"] = ces_4_gdf["ces4_dac"].fillna(0).astype(int)

# flag bay area tracts in ces_4_df
ces_4_gdf["bay_area_class"] = np.where(
    ces_4_gdf["county_name"].isin(bay_area_counties), "Bay Area", "Outside Bay Area"
 )

# summarize CES4 DAC and non-DAC counts by bay area status
ces4_summary = (
    ces_4_gdf.groupby("bay_area_class")["ces4_dac"]
    .agg(
        total_dac="sum",
        total_not_dac=lambda s: (s == 0).sum(),
        total_tracts="count",
    )
    .reset_index()
)
ces4_summary["dac_pct"] = ces4_summary["total_dac"] / ces4_summary["total_tracts"] * 100

# pivot to requested layout
ces4_pivot = (
    ces4_summary.set_index("bay_area_class")[["total_dac", "total_not_dac"]]
    .T
    .reindex(columns=["Bay Area", "Outside Bay Area"], fill_value=0)
    .astype(int)
)
ces4_pivot["total Tracts"] = ces4_pivot["Bay Area"] + ces4_pivot["Outside Bay Area"]

print("2010 tract geometries:", ca_tracts_2010["tract_geoid"].nunique())
print("tracts in CES4 table:", ces_4_df["tract_geoid"].nunique())
print("tracts in ces_4_gdf after join:", ces_4_gdf["tract_geoid"].nunique())

ces4_pivot

2010 tract geometries: 8057
tracts in CES4 table: 8035
tracts in ces_4_gdf after join: 8057


bay_area_class,Bay Area,Outside Bay Area,total Tracts
total_dac,113,1871,1984
total_not_dac,1475,4598,6073


In [31]:
# join all 2020 CA tracts with CES5 so denominator includes every 2020 tract
ces_5_gdf = ca_tracts_2020.merge(
    ces_5_df[["tract_geoid", "ces5_dac"]],
    on="tract_geoid",
    how="left",
)

# treat tracts missing from CES5 as non-DAC for full-tract denominator summaries
ces_5_gdf["ces5_dac"] = ces_5_gdf["ces5_dac"].fillna(0).astype(int)

# flag bay area tracts in ces_5_gdf
ces_5_gdf["bay_area_class"] = np.where(
    ces_5_gdf["county_name"].isin(bay_area_counties), "Bay Area", "Outside Bay Area"
 )

# summarize CES5 DAC and non-DAC counts by bay area status
ces5_summary = (
    ces_5_gdf.groupby("bay_area_class")["ces5_dac"]
    .agg(
        total_dac="sum",
        total_not_dac=lambda s: (s == 0).sum(),
        total_tracts="count",
    )
    .reset_index()
)
ces5_summary["dac_pct"] = ces5_summary["total_dac"] / ces5_summary["total_tracts"] * 100

# pivot to requested layout
ces5_pivot = (
    ces5_summary.set_index("bay_area_class")[["total_dac", "total_not_dac"]]
    .T
    .reindex(columns=["Bay Area", "Outside Bay Area"], fill_value=0)
    .astype(int)
)
ces5_pivot["total Tracts"] = ces5_pivot["Bay Area"] + ces5_pivot["Outside Bay Area"]

print("2020 tract geometries:", ca_tracts_2020["tract_geoid"].nunique())
print("tracts in CES5 table:", ces_5_df["tract_geoid"].nunique())
print("tracts in ces_5_gdf after join:", ces_5_gdf["tract_geoid"].nunique())

ces5_pivot

2020 tract geometries: 9129
tracts in CES5 table: 9106
tracts in ces_5_gdf after join: 9129


bay_area_class,Bay Area,Outside Bay Area,total Tracts
total_dac,122,2136,2258
total_not_dac,1650,5221,6871
